## Pinecone Vetor DB

In [1]:
api_key = "pcsk_BZxPg_5Q8AjNHtrMjhACS3UctgFboC4wRi3XCj9TmXXwDUUnSUEmCu4F8xT26VFgWRyr5"

In [2]:
from pinecone import Pinecone
pc = Pinecone(api_key= api_key)

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


/Users/rutwikbelsare/Rutwik/Gen AI/Projects/RagProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 45051.72it/s]


In [4]:
from pinecone import ServerlessSpec
index_name= "rag-practice"
if not pc.has_index(index_name):
    pc.create_index(name=index_name,
                    dimension=1024,
                    metric="cosine",
                    spec=ServerlessSpec(
                        cloud="aws",
                        region="us-east-1"))
index = pc.Index(index_name)

In [5]:
index

In [6]:
from langchain_pinecone import PineconeVectorStore
vector_store = PineconeVectorStore(index = index, embedding = embeddings)

In [7]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [8]:
vector_store.add_documents(documents=documents)

['a9e1b0f8-70bf-443b-b990-3a34b8d9c76a',
 'b0f4e878-51e6-4c63-8d4c-75be00573ade',
 'bf974a9d-448a-4df8-918b-9f51d6734b83',
 '65e5d6a6-3134-487e-a4f5-cb21aea5fc61',
 'b4506b5b-e738-48c9-a0c9-c1328f274e13',
 'ed3f1aa9-30d8-4eb7-9a8e-d4f2a1d89185',
 'eca8e252-fec6-46d2-a306-ce0f06a61806',
 'ad60f438-4997-44d5-863c-0bb30a585f55',
 '8314331b-49f0-44da-b103-936a0d0ae72e',
 'e8d540bf-49a1-4cc9-826f-69a0f1db08b1']

In [9]:
##Query Directly
results = vector_store.similarity_search(
    "Langchain provides abstractions to make working with LLMs easy",k=2,filter={'source':'tweet'}
)
for res in results:
    print(f"*{res.page_content} [{res.metadata}]")

*Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
*LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [11]:
results = vector_store.similarity_search_with_score("Langchain provides abstractions to make working with LLMs easy",k=2,filter = {'source':'tweet'} )

for res, score in results:
    print(f"Score: {score}, Content: {res.page_content}, Metadata: {res.metadata}")

Score: 0.642791569, Content: Building an exciting new project with LangChain - come check it out!, Metadata: {'source': 'tweet'}
Score: 0.596500278, Content: LangGraph is the best framework for building stateful, agentic applications!, Metadata: {'source': 'tweet'}


In [ ]:
retriever = vecotr_store.as_retriever()